In [1]:
import pandas as pd 

In [2]:
df = pd.read_csv(r"C:\potfolio\Fraud_Detection\data\raw\Base.csv")
df.head()

,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,...,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0,0.3,0.986506,-1,25,40,0.006735,102.453711,AA,1059,...,0,1500.0,0,INTERNET,16.224843,linux,1,1,0,0
1,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,AD,1658,...,0,1500.0,0,INTERNET,3.363854,other,1,1,0,0
2,0,0.8,0.996707,9,14,40,0.012316,-1.490386,AB,1095,...,0,200.0,0,INTERNET,22.730559,windows,0,1,0,0
3,0,0.6,0.475100,11,14,30,0.006991,-1.863101,AB,3483,...,0,200.0,0,INTERNET,15.215816,linux,1,1,0,0
4,0,0.9,0.842307,-1,29,40,5.742626,47.152498,AA,2339,...,0,200.0,0,INTERNET,3.743048,other,0,1,0,0


In [3]:
df.columns

Index(['fraud_bool', 'income', 'name_email_similarity',
       'prev_address_months_count', 'current_address_months_count',
       'customer_age', 'days_since_request', 'intended_balcon_amount',
       'payment_type', 'zip_count_4w', 'velocity_6h', 'velocity_24h',
       'velocity_4w', 'bank_branch_count_8w',
       'date_of_birth_distinct_emails_4w', 'employment_status',
       'credit_risk_score', 'email_is_free', 'housing_status',
       'phone_home_valid', 'phone_mobile_valid', 'bank_months_count',
       'has_other_cards', 'proposed_credit_limit', 'foreign_request', 'source',
       'session_length_in_minutes', 'device_os', 'keep_alive_session',
       'device_distinct_emails_8w', 'device_fraud_count', 'month'],
      dtype='object')

In [4]:
'month' in df.columns

True

In [5]:
num_col = df.drop("fraud_bool", axis=1)
num_col = num_col.select_dtypes(include=["int64","float64"]).columns


In [6]:
cat_col = df.select_dtypes(include=["object"]).columns
cat_col

Index(['payment_type', 'employment_status', 'housing_status', 'source',
       'device_os'],
      dtype='object')

In [7]:
for col in num_col:
    df[col] = df[col].fillna(df[col].mean())

In [8]:
for col in cat_col:
    df[col] = df[col].fillna("missing")


In [9]:
for col in num_col:
    print(f"missing : {df[col].isna().sum()}")

missing : 0
missing : 0
missing : 0
missing : 0
missing : 0
missing : 0
missing : 0
missing : 0
missing : 0
missing : 0
missing : 0
missing : 0
missing : 0
missing : 0
missing : 0
missing : 0
missing : 0
missing : 0
missing : 0
missing : 0
missing : 0
missing : 0
missing : 0
missing : 0
missing : 0
missing : 0


In [10]:
for col in cat_col :
    print(f"missing : {df[col].isna().sum()}")

missing : 0
missing : 0
missing : 0
missing : 0
missing : 0


**Feature engineering Base level**

In [11]:
mean = {}
std = {}

for col in num_col:
    mean[col] = df[col].mean()
    std[col] = df[col].std()
    df[col] = (df[col]- mean[col])/std[col]


In [12]:
for i in cat_col:
    print(f"count :{df[i].value_counts()} ")

count :payment_type
AB    370554
AA    258249
AC    252071
AD    118837
AE       289
Name: count, dtype: int64 
count :employment_status
CA    730252
CB    138288
CF     44034
CC     37758
CD     26522
CE     22693
CG       453
Name: count, dtype: int64 
count :housing_status
BC    372143
BB    260965
BA    169675
BE    169135
BD     26161
BF      1669
BG       252
Name: count, dtype: int64 
count :source
INTERNET    992952
TELEAPP       7048
Name: count, dtype: int64 
count :device_os
other        342728
linux        332712
windows      263506
macintosh     53826
x11            7228
Name: count, dtype: int64 


In [13]:
df = pd.get_dummies(df, columns=cat_col,drop_first=True)

In [14]:
df.head()

,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,zip_count_4w,velocity_6h,...,housing_status_BC,housing_status_BD,housing_status_BE,housing_status_BF,housing_status_BG,source_TELEAPP,device_os_macintosh,device_os_other,device_os_windows,device_os_x11
0,0,-0.904778,1.704497,-0.402272,-0.696643,0.524782,-0.189335,4.634883,-0.510946,2.469192,...,True,False,False,False,False,False,False,False,False,False
1,0,0.817325,0.427953,-0.402272,0.027285,-1.138309,-0.188711,-0.470003,0.084852,1.182299,...,True,False,False,False,False,False,False,True,False,False
2,0,0.817325,1.739778,-0.175238,-0.821068,0.524782,-0.188298,-0.501671,-0.475138,-0.396701,...,True,False,False,False,False,False,False,False,True,False
3,0,0.128484,-0.064312,-0.129831,-0.821068,-0.306764,-0.189288,-0.520089,1.900096,2.913123,...,True,False,False,False,False,False,False,False,False,False
4,0,1.161746,1.205752,-0.402272,-0.651398,0.524782,0.876452,1.902091,0.762211,0.643393,...,True,False,False,False,False,False,False,True,False,False


In [15]:
from sklearn.model_selection import train_test_split

x = df.drop("fraud_bool", axis = 1)
y = df["fraud_bool"]

x_train, x_temp , y_train, y_temp = train_test_split(x,y , test_size=0.3,stratify=y, random_state=42)

x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5,stratify=y_temp, random_state=42)

In [16]:
trian = pd.concat([x_train,y_train], axis=1)
val = pd.concat([x_val,y_val], axis = 1)
test = pd.concat([x_test, y_test], axis=1)


trian.to_csv(r"C:\potfolio\Fraud_Detection\data\processed_data\train.csv", index = False)

In [17]:
val.to_csv(r"C:\potfolio\Fraud_Detection\data\processed_data\val.csv", index = False)
test.to_csv(r"C:\potfolio\Fraud_Detection\data\processed_data\test.csv", index = False)